# YOLO26 学習用 Colab ノートブック

`shared/train.py` を Google Colab で実行するための notebook です。

前提:
- GPU ランタイムを有効化して使う
- データセットは `train/`, `valid/`, `test/` を含む YOLO 形式
- 必要なら事前学習済み重みを Google Drive に置いておく

補足:
- `WEIGHTS_PATH` を指定すると、既存の `.pt` を初期重みにした追加学習ができます
- `RESUME_TRAINING = True` にすると、中断した run の `last.pt` を使った再開を行います


In [ ]:
# Colab で GPU が見えているか確認
!nvidia-smi


In [ ]:
# Google Drive を使う場合だけ mount します
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import textwrap

# 必要に応じて変更してください
REPO_URL = "https://github.com/koshien2015/cap-baseball.git"
REPO_BRANCH = "master"
REPO_DIR = Path("/content/cap-baseball")
ULTRALYTICS_DIR = REPO_DIR / "ultralytics"
SHARED_DIR = ULTRALYTICS_DIR / "shared"

# Drive 上の学習データ配置例:
# /content/drive/MyDrive/cap-baseball/yolo-dataset/
#   ├─ train/
#   ├─ valid/
#   └─ test/
DATASET_DIR = Path("/content/drive/MyDrive/cap-baseball/yolo-dataset")

# 追加学習したい既存モデル。不要なら None。
WEIGHTS_PATH = Path("/content/drive/MyDrive/cap-baseball/weights/yolo26m.pt")
# WEIGHTS_PATH = None

# 中断した run を完全再開したい場合はこちらを使います。
# 例: /content/drive/MyDrive/cap-baseball/training-results/cap_yolo26_colab/weights/last.pt
RESUME_TRAINING = False
RESUME_WEIGHTS_PATH = Path("/content/drive/MyDrive/cap-baseball/training-results/cap_yolo26_colab/weights/last.pt")

MODEL_CFG = "yolo26m-p2.yaml"
IMGSZ = 1280
EPOCHS = 20
BATCH = 1  # VRAM に応じて -1 も可
RUN_NAME = "cap_yolo26_colab"

# 学習結果を Drive に退避したい場合の出力先
EXPORT_DIR = Path("/content/drive/MyDrive/cap-baseball/training-results")

print(f"DATASET_DIR: {DATASET_DIR}")
print(f"WEIGHTS_PATH: {WEIGHTS_PATH}")
print(f"RESUME_TRAINING: {RESUME_TRAINING}")
print(f"RESUME_WEIGHTS_PATH: {RESUME_WEIGHTS_PATH}")


## モードの使い分け

- 追加学習: `RESUME_TRAINING = False` かつ `WEIGHTS_PATH` を指定
- 完全再開: `RESUME_TRAINING = True` かつ `RESUME_WEIGHTS_PATH` に `last.pt` を指定
- 新規学習: `RESUME_TRAINING = False` かつ `WEIGHTS_PATH = None`


In [ ]:
if REPO_DIR.exists():
    print(f"Repo already exists: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(SHARED_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ULTRALYTICS_DIR)], check=True)

print("Setup complete")


In [ ]:
required_dirs = [DATASET_DIR / "train", DATASET_DIR / "valid"]
missing = [str(path) for path in required_dirs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset directories: " + ", ".join(missing))

if RESUME_TRAINING:
    if not RESUME_WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Resume weights file not found: {RESUME_WEIGHTS_PATH}")
else:
    if WEIGHTS_PATH is not None and not WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Weights file not found: {WEIGHTS_PATH}")

data_yaml_path = SHARED_DIR / "data-colab.yaml"
data_yaml_path.write_text(
    textwrap.dedent(
        f"""
        path: {DATASET_DIR.as_posix()}
        train: train
        val: valid
        test: test

        nc: 11
        names:
          0: cap
          1: pitcher_motion
          2: batter_stance
          3: umpire
          4: catcher
          5: pitcher_release
          6: batter_swing
          7: catcher_stance
          8: catcher_catch
          9: catcher_throw
          10: catcher_miss
        """
    ).strip() + "\n",
    encoding="utf-8",
)

print(data_yaml_path.read_text(encoding="utf-8"))


In [ ]:
os.chdir(SHARED_DIR)

command = [sys.executable, "train.py"]

if RESUME_TRAINING:
    command.extend([
        "--data", "data-colab.yaml",
        "--weights", str(RESUME_WEIGHTS_PATH),
        "--name", RUN_NAME,
    ])
else:
    command.extend([
        "--data", "data-colab.yaml",
        "--model", MODEL_CFG,
        "--imgsz", str(IMGSZ),
        "--epochs", str(EPOCHS),
        "--batch", str(BATCH),
        "--name", RUN_NAME,
    ])
    if WEIGHTS_PATH is not None:
        command.extend(["--weights", str(WEIGHTS_PATH)])

print(" ".join(command))
subprocess.run(command, check=True)


## 完全再開したい場合の `resume=True` 実行例

`train.py` 自体は `resume` 引数を受けていないので、optimizer state まで含めて厳密に再開したいときは次のセルを使います。
このセルは `shared/train.py` を通さず、Ultralytics API を直接呼びます。


In [ ]:
# 必要なときだけ実行
# RESUME_TRAINING = True の場合はこちらのセルを使う方が安全です

from ultralytics import YOLO

if RESUME_TRAINING:
    model = YOLO(str(RESUME_WEIGHTS_PATH))
    model.train(resume=True)
else:
    print("RESUME_TRAINING が False なのでこのセルは不要です")


In [ ]:
run_dir = SHARED_DIR / "runs" / "detect" / RUN_NAME
print(f"run_dir: {run_dir}")

if run_dir.exists():
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    export_target = EXPORT_DIR / RUN_NAME
    if export_target.exists():
        shutil.rmtree(export_target)
    shutil.copytree(run_dir, export_target)
    print(f"Exported to: {export_target}")
else:
    print("Training output directory was not found")


## 学習済みモデルのダウンロード

`best.pt` と `last.pt` があれば、Colab から直接ダウンロードできます。


In [ ]:
weights_dir = run_dir / "weights"
best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

print(f"best.pt: {best_pt} exists={best_pt.exists()}")
print(f"last.pt: {last_pt} exists={last_pt.exists()}")


In [ ]:
from google.colab import files

DOWNLOAD_BEST = True
DOWNLOAD_LAST = False

if DOWNLOAD_BEST:
    if not best_pt.exists():
        raise FileNotFoundError(f"best.pt not found: {best_pt}")
    files.download(str(best_pt))

if DOWNLOAD_LAST:
    if not last_pt.exists():
        raise FileNotFoundError(f"last.pt not found: {last_pt}")
    files.download(str(last_pt))
